# IAN 603 · Module 8
# Strings and Dates in PostgreSQL

Real-world data often arrives as messy text and inconsistent time values.

The goal is not simply to make values look nicer. The goal is to transform them **without losing meaning or traceability**.

Use this workflow:

```text
inspect raw values
      ↓
normalize representation
      ↓
validate structure
      ↓
parse into the correct type
      ↓
calculate / group / compare
      ↓
format only for display
      ↓
validate the transformation
```

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module8.html


## What you will learn

By the end of this notebook, you should be able to:

1. Inspect messy text before transforming it.
2. Normalize case, boundaries, and repeated whitespace.
3. Use `LENGTH` to reveal hidden spaces.
4. Combine fields safely with `CONCAT_WS`.
5. Parse structured strings with `SPLIT_PART`, `POSITION`, and `SUBSTRING`.
6. Use `REPLACE` and regular expressions appropriately.
7. Distinguish **cleaning** from **validation**.
8. Explain the differences among `DATE`, `TIMESTAMP`, `TIMESTAMPTZ`, and `INTERVAL`.
9. Extract date/time components.
10. Use `DATE_TRUNC` for analytical time buckets.
11. Calculate elapsed time from timestamps.
12. Add intervals to timestamps.
13. Distinguish `AGE` from exact elapsed time.
14. Validate text before parsing it into a date.
15. Use `TO_DATE` and casts appropriately.
16. Format values with `TO_CHAR` only after analysis.
17. Convert `TIMESTAMPTZ` values across time zones.
18. Preserve raw values when transformations are interpretive.


## 0. Install required Python packages

Run this cell if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** The helper is downloaded automatically when needed.

If you want your own copy:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)


In [ ]:
from pathlib import Path
import urllib.request

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link above, then run this cell again.")
        raise

from ian603_database import IAN603Database

db = IAN603Database()
display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window.

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password and click **Connect**.


In [ ]:
db.show_connection_panel()


### Optional: Connect manually if the connection window does not work

If the PostgreSQL connection window above does not display, or the **Connect** button does not work in your Python/Jupyter environment, run the next cell instead.

You only need to use **one** connection method. The password prompt in the manual option is hidden while you type.


In [ ]:
from getpass import getpass

print("Manual PostgreSQL Connection")
print("Press Enter to use the default value shown in brackets.\n")

host = input("Host [localhost]: ").strip() or "localhost"
database = input("Database [postgres]: ").strip() or "postgres"
user = input("Username [postgres]: ").strip() or "postgres"

port_text = input("Port [5432]: ").strip()
port = int(port_text) if port_text else 5432

password = getpass("PostgreSQL password: ")

manual_config = {
    "host": host,
    "database": database,
    "user": user,
    "port": port,
    "password": password,
}

db.connect(
    config=manual_config,
    save_settings=False,
    save_password=False,
    verbose=True,
)


## 3. Check the connection

After clicking **Connect**, run this cell once.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


# Part I · Meet the Teaching Data

This module uses deliberately inconsistent text and several temporal data types.

| Table | What it contains |
|---|---|
| `patients` | names, emails, addresses, admission dates |
| `admissions` | admission and discharge timestamps |
| `products` | product names and descriptions |
| `customers` | customer text plus signup timestamps |
| `orders` | time-zone-aware order timestamps |
| `staging_dates` | raw date/timestamp text |

The important idea is that **raw text is evidence**.

Do not overwrite or discard it until you understand what the transformation means.


## 4. Create or reset the Module 8 teaching database


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 8\n-- Strings and Dates in PostgreSQL\n-- ============================================================\n-- Companion file for s_module8.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- Topics:\n-- trimming, case normalization, splitting, replacing, regex,\n-- date parsing, DATE_TRUNC, intervals, AGE, formatting,\n-- duration calculations, and time-zone reasoning.\n--\n-- Preserve raw values when a transformation is interpretive,\n-- and validate before converting messy text into typed values.\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET TEACHING DATABASE\n-- ============================================================\n\nDROP TABLE IF EXISTS staging_dates CASCADE;\nDROP TABLE IF EXISTS orders CASCADE;\nDROP TABLE IF EXISTS customers CASCADE;\nDROP TABLE IF EXISTS products CASCADE;\nDROP TABLE IF EXISTS admissions CASCADE;\nDROP TABLE IF EXISTS patients CASCADE;\n\nCREATE TABLE patients (\n    subject_id integer PRIMARY KEY,\n    name text NOT NULL,\n    email text,\n    address text,\n    admission_date date\n);\n\nCREATE TABLE admissions (\n    hadm_id integer PRIMARY KEY,\n    subject_id integer NOT NULL REFERENCES patients(subject_id),\n    admit_time timestamp NOT NULL,\n    discharge_time timestamp,\n    diagnosis text NOT NULL,\n    CHECK (discharge_time IS NULL OR discharge_time >= admit_time)\n);\n\nCREATE TABLE products (\n    product_id integer PRIMARY KEY,\n    product_name text NOT NULL,\n    description text,\n    price numeric(10,2) NOT NULL CHECK (price >= 0)\n);\n\nCREATE TABLE customers (\n    customer_id integer PRIMARY KEY,\n    name text NOT NULL,\n    email text,\n    address text,\n    signup_at timestamp NOT NULL\n);\n\nCREATE TABLE orders (\n    order_id integer PRIMARY KEY,\n    customer_id integer NOT NULL REFERENCES customers(customer_id),\n    order_ts timestamptz NOT NULL,\n    total_amount numeric(10,2) NOT NULL CHECK (total_amount >= 0)\n);\n\nCREATE TABLE staging_dates (\n    row_id integer PRIMARY KEY,\n    raw_date text,\n    raw_timestamp text\n);\n\nINSERT INTO patients VALUES\n(1001,'Alice Smith','alice.smith@email.com','123 Main St, Greensboro',DATE '2026-01-10'),\n(1002,'bob JOHNSON','BOB.JOHNSON@EMAIL.COM','456 Oak Ave, greensboro',DATE '2026-02-01'),\n(1003,'  Carol   White  ','carol.white@email.com','789 Pine Rd, GSO',DATE '2026-01-20'),\n(1004,'DAVID  Brown  ',' david.brown@email.com ','101 Elm St, Greensboro',DATE '2026-02-15'),\n(1005,'EVE DAVIS',NULL,'202 Maple Dr, GSO',DATE '2026-03-02'),\n(1006,'Frank Miller','frank.miller@sample.org','  88 Market St, High Point  ',DATE '2026-05-11');\n\nINSERT INTO admissions VALUES\n(101,1001,TIMESTAMP '2026-01-10 08:15:00',TIMESTAMP '2026-01-15 14:30:00','Pneumonia'),\n(102,1002,TIMESTAMP '2026-02-01 19:42:00',TIMESTAMP '2026-02-03 10:10:00','Fracture'),\n(103,1001,TIMESTAMP '2026-03-05 09:05:00',TIMESTAMP '2026-03-12 11:45:00','Hypertension'),\n(104,1003,TIMESTAMP '2026-01-20 23:10:00',TIMESTAMP '2026-01-25 09:00:00','Infection'),\n(105,1004,TIMESTAMP '2026-02-15 06:35:00',TIMESTAMP '2026-02-20 16:20:00','Surgery'),\n(106,1005,TIMESTAMP '2026-04-20 13:00:00',NULL,'Observation');\n\nINSERT INTO products VALUES\n(1,'Laptop','High-performance laptop',999.99),\n(2,'MOUSE','Wireless Mouse',29.99),\n(3,'Keyboard','Mechanical KB',79.99),\n(4,'Monitor','27-inch 4K monitor',399.99),\n(5,'  Headphones  ','Noise-cancelling headphones',199.99);\n\nINSERT INTO customers VALUES\n(1,'John Doe','john.doe@email.com','123 Main St, Greensboro',TIMESTAMP '2025-11-15 10:00:00'),\n(2,'JANE SMITH','jane.smith@email.com','456 Oak Ave, High Point',TIMESTAMP '2025-12-20 15:30:00'),\n(3,'  Bob Johnson ','bob.johnson@email.com','789 Pine Rd, Greensboro',TIMESTAMP '2026-01-10 09:20:00'),\n(4,'Maya Patel','maya.patel@example.org','55 Spring St, Burlington',TIMESTAMP '2026-02-14 12:00:00');\n\nINSERT INTO orders VALUES\n(201,1,TIMESTAMPTZ '2026-01-15 10:20:00-05',150.00),\n(202,2,TIMESTAMPTZ '2026-02-20 16:05:00-05',300.00),\n(203,1,TIMESTAMPTZ '2026-03-10 09:15:00-05',200.00),\n(204,3,TIMESTAMPTZ '2026-01-25 20:10:00-05',450.00),\n(205,2,TIMESTAMPTZ '2026-02-28 08:00:00-05',125.00),\n(206,4,TIMESTAMPTZ '2026-03-18 14:40:00-04',510.00),\n(207,1,TIMESTAMPTZ '2026-04-05 11:25:00-04',80.00),\n(208,3,TIMESTAMPTZ '2026-04-22 17:55:00-04',225.00);\n\nINSERT INTO staging_dates VALUES\n(1,'2026-08-01','2026-08-01 14:30:00'),\n(2,'08/05/2026','08/05/2026 09:15'),\n(3,'20260807','20260807 1645'),\n(4,'',''),\n(5,'not-a-date','not-a-timestamp');"

execute_script(setup_sql)


# Part II · Inspect Raw Text First

Before calling `UPPER`, `LOWER`, `TRIM`, or regex functions, look at the actual values.

Messy text may contain:

- leading spaces,
- trailing spaces,
- repeated internal spaces,
- inconsistent case,
- abbreviations,
- missing values,
- invalid structure.


## 5. Inspect patient text exactly as stored


In [ ]:
display_query("""
SELECT
    subject_id,
    name,
    email,
    address
FROM patients
ORDER BY subject_id;
""", title="Raw patient text")


## 6. Make hidden spaces visible with LENGTH

A value may look normal on screen while containing extra whitespace.

Compare:

- `LENGTH(name)`
- `LENGTH(BTRIM(name))`

If the numbers differ, there were boundary spaces.


In [ ]:
display_query("""
SELECT
    subject_id,
    name,
    LENGTH(name) AS raw_length,
    LENGTH(BTRIM(name)) AS trimmed_length,
    LENGTH(name) - LENGTH(BTRIM(name)) AS boundary_space_count
FROM patients
ORDER BY subject_id;
""", title="Detect boundary whitespace")


# Part III · Normalize Case and Boundaries

Common functions:

| Function | Purpose |
|---|---|
| `UPPER(text)` | uppercase |
| `LOWER(text)` | lowercase |
| `INITCAP(text)` | capitalize word starts |
| `BTRIM(text)` | remove whitespace at both ends |
| `LTRIM(text)` | remove left-side whitespace |
| `RTRIM(text)` | remove right-side whitespace |

These functions change **representation**, not truth.


## 7. Compare case and trimming transformations


In [ ]:
display_query("""
SELECT
    subject_id,
    name AS raw_name,
    UPPER(name) AS upper_name,
    LOWER(name) AS lower_name,
    INITCAP(BTRIM(name)) AS display_name,
    LENGTH(name) AS raw_length,
    LENGTH(BTRIM(name)) AS trimmed_length
FROM patients
ORDER BY subject_id;
""", title="Case, trim, and length")


### Important limitation

`BTRIM` removes whitespace at the beginning and end.

It does **not** fix repeated spaces inside the string.

Example:

```text
'  Carol   White  '
      ↓ BTRIM
'Carol   White'
```

The three spaces between the names still remain.


# Part IV · Normalize Repeated Internal Whitespace

A regular expression can collapse repeated whitespace:

```sql
REGEXP_REPLACE(
    BTRIM(name),
    '[[:space:]]+',
    ' ',
    'g'
)
```

Read it as:

1. trim the boundaries,
2. find one-or-more whitespace characters,
3. replace each run with one space,
4. repeat globally.


## 8. Normalize names step by step


In [ ]:
display_query("""
SELECT
    subject_id,
    name AS raw_name,
    BTRIM(name) AS boundary_trimmed,
    REGEXP_REPLACE(
        BTRIM(name),
        '[[:space:]]+',
        ' ',
        'g'
    ) AS whitespace_normalized,
    INITCAP(
        REGEXP_REPLACE(
            BTRIM(name),
            '[[:space:]]+',
            ' ',
            'g'
        )
    ) AS display_name
FROM patients
ORDER BY subject_id;
""", title="Name normalization pipeline")


### Before / after reasoning

| Raw | Normalized |
|---|---|
| `'  Carol   White  '` | `'Carol White'` |
| `'DAVID  Brown  '` | `'David Brown'` |

Notice that normalization is a **rule**. It may be appropriate for names here, but not for every text column.


# Part V · Combine Fields Carefully

PostgreSQL `CONCAT_WS` means:

> concatenate with separator

It is useful because NULL arguments are skipped.

For a contact line:

```sql
CONCAT_WS(' | ', name, email, address)
```

can avoid awkward repeated separators caused by NULL values.


## 9. Build a readable contact line


In [ ]:
display_query("""
SELECT
    subject_id,
    CONCAT_WS(
        ' | ',
        INITCAP(
            REGEXP_REPLACE(
                BTRIM(name),
                '[[:space:]]+',
                ' ',
                'g'
            )
        ),
        NULLIF(BTRIM(email), ''),
        BTRIM(address)
    ) AS contact_line
FROM patients
ORDER BY subject_id;
""", title="Contact line")


### Why use NULLIF here?

`CONCAT_WS` skips NULL, but an empty string is still a value.

```sql
NULLIF(BTRIM(email), '')
```

converts blank email text to NULL before concatenation.


# Part VI · Parse Structured Strings

Some text has an internal structure.

Example email:

```text
alice.smith@email.com
```

Conceptually:

```text
alice.smith | @ | email.com
 user            domain
```

PostgreSQL offers several tools for this kind of parsing.


## 10. SPLIT_PART

```sql
SPLIT_PART(text, delimiter, field_number)
```

For email:

- field 1 → user
- field 2 → domain


In [ ]:
display_query("""
SELECT
    subject_id,
    email,
    SPLIT_PART(
        LOWER(BTRIM(email)),
        '@',
        1
    ) AS email_user,
    SPLIT_PART(
        LOWER(BTRIM(email)),
        '@',
        2
    ) AS email_domain
FROM patients
ORDER BY subject_id;
""", title="Parse email components")


## 11. POSITION answers “where?”

```sql
POSITION('@' IN email)
```

returns the character position of the first match.

A zero result means the character was not found.


In [ ]:
display_query("""
SELECT
    subject_id,
    email,
    POSITION('@' IN COALESCE(email, '')) AS at_position
FROM patients
ORDER BY subject_id;
""", title="Find the @ position")


## 12. SUBSTRING extracts a region

```sql
SUBSTRING(text FROM start FOR length)
```

This is useful for reproducible codes when the rule is explicit.


In [ ]:
display_query("""
SELECT
    product_id,
    product_name,
    UPPER(
        SUBSTRING(
            BTRIM(product_name)
            FROM 1 FOR 3
        )
    )
    || '-'
    || LPAD(product_id::text, 3, '0') AS product_code
FROM products
ORDER BY product_id;
""", title="Reproducible product codes")


# Part VII · REPLACE vs Regular Expressions

Use the simplest tool that solves the problem.

### REPLACE

Best when the exact text to replace is known.

### REGEXP_REPLACE

Best when the pattern is variable or structural.


## 13. Exact replacement with REPLACE


In [ ]:
display_query("""
SELECT
    product_id,
    description,
    REPLACE(
        description,
        'KB',
        'keyboard'
    ) AS expanded_description
FROM products
ORDER BY product_id;
""", title="Exact text replacement")


## 14. Pattern-based cleanup with REGEXP_REPLACE

This teaching example removes characters outside an allowed email character set.

Important:

> Cleaning a string into a plausible form does **not** prove the address is valid.


In [ ]:
display_query("""
SELECT
    subject_id,
    email AS raw_email,
    LOWER(
        REGEXP_REPLACE(
            BTRIM(email),
            '[^[:alnum:]@._+-]',
            '',
            'g'
        )
    ) AS cleaned_email
FROM patients
WHERE email IS NOT NULL
ORDER BY subject_id;
""", title="Teaching-level email cleanup")


# Part VIII · Cleaning Is Not Validation

These are different tasks.

### Cleaning

> Transform representation.

Examples:
- trim spaces,
- standardize case,
- remove known punctuation.

### Validation

> Test whether a value satisfies an expected rule.

A cleaned email can still be structurally invalid.


## 15. Check email shape with a regular expression


In [ ]:
display_query("""
SELECT
    subject_id,
    email,
    CASE
        WHEN LOWER(BTRIM(email))
             ~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$'
            THEN 'valid shape'
        ELSE 'review'
    END AS email_shape_check
FROM patients
ORDER BY subject_id;
""", title="Email shape validation")


### Important wording

This is a **shape check**, not proof that the email exists.

Regex can tell us:

> the string resembles the pattern we expect.

It cannot tell us:

> the mailbox is real and reachable.


# Part IX · Text-Function Decision Guide

| Need | Useful function |
|---|---|
| remove boundary spaces | `BTRIM` |
| normalize case | `LOWER`, `UPPER`, `INITCAP` |
| measure text | `LENGTH` |
| split by delimiter | `SPLIT_PART` |
| find character position | `POSITION` |
| take part of string | `SUBSTRING` |
| exact replacement | `REPLACE` |
| pattern replacement | `REGEXP_REPLACE` |
| pattern validation | regex match `~` |
| combine nullable fields | `CONCAT_WS` |

Start simple. Use regex only when a simpler text function is not enough.


# Part X · PostgreSQL Temporal Types

Choosing the correct type is part of data modeling.

| Type | Represents |
|---|---|
| `DATE` | calendar date |
| `TIMESTAMP` | date + clock time, no time-zone context |
| `TIMESTAMPTZ` | an instant in time, displayed using a time zone |
| `INTERVAL` | duration / amount of time |

These types answer different questions.


## 16. Inspect the temporal column types

Use the PostgreSQL catalog to confirm what is actually stored.


In [ ]:
display_query("""
SELECT
    table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name IN ('patients', 'admissions', 'customers', 'orders')
  AND (
      data_type LIKE '%date%'
      OR data_type LIKE '%timestamp%'
  )
ORDER BY table_name, ordinal_position;
""", title="Temporal column types")


### A practical distinction

`admissions.admit_time` is a `timestamp`.

`orders.order_ts` is a `timestamptz`.

That difference matters when converting between local and UTC clock representations later.


# Part XI · EXTRACT: Pull Out One Component

`EXTRACT` returns one part of a date or timestamp.

Examples:

- year,
- month,
- day,
- hour,
- day of week.


## 17. Extract admission components


In [ ]:
display_query("""
SELECT
    hadm_id,
    admit_time,
    EXTRACT(YEAR FROM admit_time) AS year,
    EXTRACT(MONTH FROM admit_time) AS month,
    EXTRACT(DOW FROM admit_time) AS dow,
    EXTRACT(HOUR FROM admit_time) AS hour
FROM admissions
ORDER BY admit_time;
""", title="Admission time components")


### Day-of-week caution

In PostgreSQL:

`EXTRACT(DOW FROM ...)`

uses:

| Value | Day |
|---:|---|
| 0 | Sunday |
| 1 | Monday |
| ... | ... |
| 6 | Saturday |

Do not assume every database or programming language uses the same numbering.


# Part XII · DATE_TRUNC: Create Time Buckets

`DATE_TRUNC` returns a timestamp aligned to the beginning of a time unit.

Examples:

- day,
- week,
- month,
- quarter,
- year.

This is especially useful for analytical grouping.


## 18. Truncate order timestamps to several levels


In [ ]:
display_query("""
SELECT
    order_id,
    order_ts,
    DATE_TRUNC('day', order_ts) AS order_day,
    DATE_TRUNC('week', order_ts) AS order_week,
    DATE_TRUNC('month', order_ts) AS order_month,
    DATE_TRUNC('quarter', order_ts) AS order_quarter
FROM orders
ORDER BY order_ts;
""", title="DATE_TRUNC levels")


## EXTRACT vs DATE_TRUNC

These are not interchangeable.

### EXTRACT(MONTH ...)

Returns only the number:

```text
1
2
3
```

That loses the year.

### DATE_TRUNC('month', ...)

Returns a month boundary such as:

```text
2026-01-01
2026-02-01
```

For multi-year monthly analysis, `DATE_TRUNC` is usually safer.


# Part XIII · Monthly Aggregation

Once timestamps are placed into consistent month buckets, we can aggregate.

Source grain:

> one row per order

Target grain:

> **one row per month**


## 19. Monthly order summary


In [ ]:
display_query("""
SELECT
    DATE_TRUNC('month', order_ts)::date AS month,
    COUNT(*) AS order_count,
    COUNT(DISTINCT customer_id) AS customers,
    SUM(total_amount) AS revenue,
    ROUND(AVG(total_amount), 2) AS avg_order
FROM orders
GROUP BY DATE_TRUNC('month', order_ts)
ORDER BY month;
""", title="Monthly order summary")


### Grain changed

Before:

> one row per order

After:

> one row per month

The time transformation and the aggregation work together.


# Part XIV · Timestamp Subtraction

Subtracting timestamps returns an `INTERVAL`.

For a hospital admission:

```text
discharge_time - admit_time
```

produces the elapsed interval.


## 20. Calculate exact stay intervals


In [ ]:
display_query("""
SELECT
    hadm_id,
    subject_id,
    admit_time,
    discharge_time,
    discharge_time - admit_time AS stay_interval
FROM admissions
WHERE discharge_time IS NOT NULL
ORDER BY hadm_id;
""", title="Length-of-stay intervals")


# Part XV · Convert an Interval to Numeric Days

Sometimes analysis needs a numeric measure rather than an interval.

A common PostgreSQL pattern is:

```sql
EXTRACT(EPOCH FROM interval) / 86400.0
```

because one day contains 86,400 seconds.


## 21. Convert length of stay to days


In [ ]:
display_query("""
SELECT
    hadm_id,
    subject_id,
    discharge_time - admit_time AS stay_interval,
    ROUND(
        EXTRACT(
            EPOCH FROM (discharge_time - admit_time)
        ) / 86400.0,
        2
    ) AS stay_days
FROM admissions
WHERE discharge_time IS NOT NULL
ORDER BY stay_days DESC;
""", title="Stay interval and numeric days")


### Why keep both?

The interval is semantically rich.

The numeric days value is convenient for:
- averages,
- thresholds,
- charts,
- models.

Keeping both during exploration makes the transformation easier to audit.


# Part XVI · Add Intervals to Dates and Timestamps

Intervals are also useful for scheduling future points in time.

Example:

```sql
discharge_time + INTERVAL '14 days'
```


## 22. Create follow-up timestamps


In [ ]:
display_query("""
SELECT
    hadm_id,
    discharge_time,
    discharge_time + INTERVAL '14 days' AS followup_14d,
    discharge_time + INTERVAL '30 days' AS followup_30d
FROM admissions
WHERE discharge_time IS NOT NULL
ORDER BY discharge_time;
""", title="Follow-up timestamps")


# Part XVII · AGE vs Exact Elapsed Time

PostgreSQL `AGE(later, earlier)` returns a calendar-style interval.

Direct subtraction returns exact elapsed time.

These answer slightly different questions.


## 23. Compare AGE with subtraction


In [ ]:
display_query("""
SELECT
    hadm_id,
    AGE(discharge_time, admit_time) AS calendar_interval,
    discharge_time - admit_time AS exact_elapsed_interval
FROM admissions
WHERE discharge_time IS NOT NULL
ORDER BY hadm_id;
""", title="AGE vs exact elapsed interval")


### When might AGE be useful?

- human-readable calendar differences,
- years/months/day style interpretation.

### When is subtraction often better?

- exact elapsed duration,
- hours,
- minutes,
- numeric stay calculations.


# Part XVIII · Parsing Is Different from Formatting

### Parsing

Convert text into a typed value.

```text
'03/15/2026'
      ↓
DATE
```

### Formatting

Convert a typed value into presentation text.

```text
DATE / TIMESTAMP
      ↓
'March 15, 2026'
```

Do analysis with typed dates and timestamps.

Format near the end for display.


# Part XIX · Validate Before Parsing

The staging table intentionally contains multiple date-text formats and bad values.

Do not immediately cast everything.

First inspect what formats are actually present.


## 24. Inspect raw date strings


In [ ]:
display_query("""
SELECT
    row_id,
    raw_date,
    raw_timestamp
FROM staging_dates
ORDER BY row_id;
""", title="Raw staging date text")


## 25. Profile the raw date values

This helps reveal repeated formats and unusual values before we write parsing rules.


In [ ]:
display_query("""
SELECT
    raw_date,
    COUNT(*) AS occurrences
FROM staging_dates
GROUP BY raw_date
ORDER BY occurrences DESC, raw_date;
""", title="Raw date profile")


# Part XX · Safe Date Parsing

A robust teaching pattern is:

```text
blank?
  ↓
expected ISO pattern?
  ↓
expected MM/DD/YYYY pattern?
  ↓
expected compact YYYYMMDD pattern?
  ↓
otherwise review
```

This makes assumptions visible.


## 26. Parse accepted date formats


In [ ]:
display_query("""
SELECT
    row_id,
    raw_date,
    CASE
        WHEN NULLIF(BTRIM(raw_date), '') IS NULL
            THEN NULL

        WHEN raw_date ~ '^\d{4}-\d{2}-\d{2}$'
            THEN raw_date::date

        WHEN raw_date ~ '^\d{2}/\d{2}/\d{4}$'
            THEN TO_DATE(raw_date, 'MM/DD/YYYY')

        WHEN raw_date ~ '^\d{8}$'
            THEN TO_DATE(raw_date, 'YYYYMMDD')

        ELSE NULL
    END AS parsed_date
FROM staging_dates
ORDER BY row_id;
""", title="Safely parsed dates")


### Why not just cast everything?

A direct cast such as:

```sql
raw_date::date
```

expects PostgreSQL to interpret every row as a valid date in an accepted format.

One invalid row can fail the query.

Validation before parsing gives us a controlled path.


## 27. Find values that need review


In [ ]:
display_query("""
SELECT
    row_id,
    raw_date
FROM staging_dates
WHERE NULLIF(BTRIM(raw_date), '') IS NOT NULL
  AND raw_date !~ '^\d{4}-\d{2}-\d{2}$'
  AND raw_date !~ '^\d{2}/\d{2}/\d{4}$'
  AND raw_date !~ '^\d{8}$'
ORDER BY row_id;
""", title="Unrecognized date-text values")


# Part XXI · TO_CHAR Is for Display

`TO_CHAR` converts typed date/time values to formatted text.

Use it after calculations and comparisons, not as your main analytical representation.


## 28. Format admission timestamps


In [ ]:
display_query("""
SELECT
    hadm_id,
    admit_time,
    TO_CHAR(
        admit_time,
        'FMMonth DD, YYYY HH24:MI'
    ) AS formatted_admit,
    TO_CHAR(
        admit_time,
        'YYYY-MM-DD'
    ) AS iso_date_text
FROM admissions
ORDER BY admit_time;
""", title="Formatted admission timestamps")


### Why formatting comes late

Typed timestamps can be:

- compared,
- sorted chronologically,
- subtracted,
- truncated,
- shifted by intervals.

Formatted text is mainly for people to read.

Keep the typed value for analysis.


# Part XXII · TIMESTAMP vs TIMESTAMPTZ

A `timestamp` is a date and clock time without time-zone context.

A `timestamptz` represents an instant in time.

PostgreSQL may display a `timestamptz` using the session time zone, but the underlying meaning is an absolute instant.


## 29. Inspect the order timestamps

The sample order rows include offsets such as `-05` and `-04`, demonstrating daylight-saving differences.


In [ ]:
display_query("""
SELECT
    order_id,
    order_ts,
    total_amount
FROM orders
ORDER BY order_id;
""", title="TIMESTAMPTZ order values")


# Part XXIII · AT TIME ZONE

For a `timestamptz`, this pattern:

```sql
order_ts AT TIME ZONE 'America/New_York'
```

returns the local wall-clock representation in that zone.

The same instant can be shown as different clock times in different zones.


## 30. Compare Eastern and UTC clock times


In [ ]:
display_query("""
SELECT
    order_id,
    order_ts,
    order_ts AT TIME ZONE 'America/New_York' AS eastern_local,
    order_ts AT TIME ZONE 'UTC' AS utc_clock
FROM orders
ORDER BY order_id;
""", title="Time-zone conversion")


### Key idea

```text
one instant
   ↓
Eastern clock representation
   ↓
UTC clock representation
```

The instant did not change.

Only the clock representation changed.


# Part XXIV · Combined Patient Profile

Now combine several string and date transformations into one analytical result.

Target grain:

> **one row per admission**


In [ ]:
display_query("""
SELECT
    p.subject_id,
    INITCAP(
        REGEXP_REPLACE(
            BTRIM(p.name),
            '[[:space:]]+',
            ' ',
            'g'
        )
    ) AS clean_name,
    SPLIT_PART(
        LOWER(BTRIM(p.email)),
        '@',
        2
    ) AS email_domain,
    a.hadm_id,
    DATE_TRUNC(
        'month',
        a.admit_time
    )::date AS admit_month,
    ROUND(
        EXTRACT(
            EPOCH FROM (a.discharge_time - a.admit_time)
        ) / 86400.0,
        2
    ) AS stay_days
FROM patients AS p
JOIN admissions AS a USING (subject_id)
ORDER BY p.subject_id, a.admit_time;
""", title="Clean patient-admission profile")


### Notice the sequence

1. raw patient name → normalized display name
2. raw email → normalized domain
3. admission timestamp → month bucket
4. two timestamps → elapsed stay days

Each transformation has a specific analytical purpose.


# Part XXV · Contact Data Quality Example

This example deliberately keeps both raw and transformed values.

That makes the cleaning process auditable.


In [ ]:
display_query("""
SELECT
    subject_id,
    name AS raw_name,
    INITCAP(
        REGEXP_REPLACE(
            BTRIM(name),
            '[[:space:]]+',
            ' ',
            'g'
        )
    ) AS clean_name,
    email AS raw_email,
    LOWER(BTRIM(email)) AS clean_email,
    SPLIT_PART(
        LOWER(BTRIM(email)),
        '@',
        2
    ) AS email_domain,
    CASE
        WHEN email IS NULL THEN 'missing'
        WHEN LOWER(BTRIM(email))
             ~ '^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$'
            THEN 'valid shape'
        ELSE 'review'
    END AS email_status,
    address AS raw_address,
    BTRIM(
        REPLACE(
            address,
            'GSO',
            'Greensboro'
        )
    ) AS clean_address
FROM patients
ORDER BY subject_id;
""", title="Contact-data quality profile")


### Raw + cleaned is better than cleaned alone

Keeping both columns lets you answer:

- What did the source contain?
- What rule changed it?
- Was the rule appropriate?
- Can the transformation be audited later?


# Part XXVI · Monthly Order Analysis

This section combines:

- `TIMESTAMPTZ`,
- `DATE_TRUNC`,
- grouping,
- counts,
- revenue,
- average order value.


In [ ]:
display_query("""
SELECT
    DATE_TRUNC('month', order_ts)::date AS month,
    COUNT(*) AS orders,
    COUNT(DISTINCT customer_id) AS customers,
    SUM(total_amount) AS revenue,
    ROUND(AVG(total_amount), 2) AS avg_order
FROM orders
GROUP BY DATE_TRUNC('month', order_ts)
ORDER BY month;
""", title="Monthly order analysis")


# Part XXVII · Validation Habits

Transformation queries need validation just like joins and aggregations.


## 31. Compare raw and normalized names

A good validation result keeps both values visible.


In [ ]:
display_query("""
SELECT
    subject_id,
    name AS raw_name,
    BTRIM(name) AS trimmed_name,
    REGEXP_REPLACE(
        BTRIM(name),
        '[[:space:]]+',
        ' ',
        'g'
    ) AS whitespace_normalized
FROM patients
ORDER BY subject_id;
""", title="Raw vs normalized names")


## 32. Validate email-domain extraction

A domain should generally be present only when the source contains a usable `@` structure.


In [ ]:
display_query("""
SELECT
    subject_id,
    email,
    POSITION('@' IN COALESCE(email, '')) AS at_position,
    SPLIT_PART(
        LOWER(BTRIM(email)),
        '@',
        2
    ) AS email_domain
FROM patients
ORDER BY subject_id;
""", title="Validate email parsing")


## 33. Validate monthly totals

Grouping by month should preserve total revenue.


In [ ]:
display_query("""
WITH source AS (
    SELECT SUM(total_amount) AS source_revenue
    FROM orders
),
monthly AS (
    SELECT
        DATE_TRUNC('month', order_ts) AS month,
        SUM(total_amount) AS monthly_revenue
    FROM orders
    GROUP BY DATE_TRUNC('month', order_ts)
)
SELECT
    (SELECT source_revenue FROM source) AS source_revenue,
    SUM(monthly_revenue) AS grouped_revenue,
    (SELECT source_revenue FROM source)
        = SUM(monthly_revenue) AS totals_match
FROM monthly;
""", title="Monthly revenue validation")


### Validation checklist

After a string/date transformation, ask:

1. Did I inspect the raw value first?
2. Did I preserve raw data when the transformation is interpretive?
3. Did I normalize only what the rule justifies?
4. Did I validate before parsing?
5. Did any NULLs appear unexpectedly?
6. Did my time bucket change the grain as intended?
7. Did totals remain consistent after grouping?
8. Am I using typed dates for analysis and formatted text only for display?
9. Is time-zone context explicit?


# Part XXVIII · Guided Practice

For each task, describe the transformation in plain language before writing SQL.


## Practice 1 · Normalize names and email addresses

Return:

- subject ID
- raw name
- cleaned display name
- raw email
- cleaned lowercase email

<details>
<summary>Hint</summary>

For names, consider:

`BTRIM → REGEXP_REPLACE → INITCAP`

For email, consider:

`BTRIM → LOWER`

</details>


In [ ]:
practice_1 = """
SELECT
    subject_id,
    name AS raw_name,
    -- clean name,
    email AS raw_email,
    -- clean email
FROM patients
ORDER BY subject_id;
"""

# display_query(practice_1, title="Practice 1")


## Practice 2 · Extract email domains

Return:

- subject ID
- email
- domain

Also include the position of `@` so you can validate the extraction.


In [ ]:
practice_2 = """
SELECT
    subject_id,
    email,
    -- @ position,
    -- domain
FROM patients
ORDER BY subject_id;
"""


## Practice 3 · Parse date text safely

Support these formats:

- `YYYY-MM-DD`
- `MM/DD/YYYY`
- `YYYYMMDD`

Anything else should become NULL for later review.

Do not blindly cast every row.


In [ ]:
practice_3 = """
SELECT
    row_id,
    raw_date,
    CASE
        -- parsing rules
    END AS parsed_date
FROM staging_dates
ORDER BY row_id;
"""


## Practice 4 · Monthly order summary

Create one row per month with:

- order count
- distinct customers
- revenue
- average order value

Use `DATE_TRUNC`.


In [ ]:
practice_4 = """
SELECT
    -- month bucket,
    -- metrics
FROM orders
GROUP BY
    -- same month expression
ORDER BY
    -- month
;
"""


## Practice 5 · Length of stay

For completed admissions, return:

- admission ID
- stay interval
- stay hours
- stay days

<details>
<summary>Hint</summary>

Use `EXTRACT(EPOCH FROM interval)`.

- divide by 3600 for hours
- divide by 86400 for days

</details>


In [ ]:
practice_5 = """
SELECT
    hadm_id,
    -- interval,
    -- hours,
    -- days
FROM admissions
WHERE discharge_time IS NOT NULL
ORDER BY hadm_id;
"""


## Practice 6 · Time zones

For each order, show:

- stored `TIMESTAMPTZ`
- Eastern local clock time
- UTC clock time

Then explain in one sentence whether these represent different events or different representations of the same event.


In [ ]:
practice_6 = """
SELECT
    order_id,
    order_ts,
    -- Eastern,
    -- UTC
FROM orders
ORDER BY order_id;
"""


# Part XXIX · Final Challenge

Produce **one row per admission** with:

- subject ID
- cleaned patient name
- email domain
- admission month
- formatted admission label
- exact stay interval
- stay hours
- stay days

### Requirements

1. Preserve admission grain.
2. Use a cleaned display name.
3. Extract the email domain from normalized email text.
4. Use `DATE_TRUNC` for the admission month.
5. Use `TO_CHAR` only for the display label.
6. Calculate stay from typed timestamps.
7. Keep open admissions in the result and let stay calculations remain NULL when discharge time is missing.


In [ ]:
final_query = """
SELECT
    p.subject_id,
    -- cleaned patient name,
    -- email domain,
    a.hadm_id,
    -- admission month,
    -- formatted admission label,
    -- exact stay interval,
    -- stay hours,
    -- stay days
FROM patients AS p
JOIN admissions AS a USING (subject_id)
ORDER BY a.admit_time;
"""

# display_query(final_query, title="Final challenge")


# Quick Concept Check

### What does BTRIM remove?
<details><summary>Show answer</summary>Whitespace from both ends of a string.</details>

### Why might LENGTH(name) differ from LENGTH(BTRIM(name))?
<details><summary>Show answer</summary>The raw string contains leading or trailing whitespace.</details>

### What is the difference between cleaning and validation?
<details><summary>Show answer</summary>Cleaning changes representation; validation tests whether a value satisfies an expected rule.</details>

### What does DATE_TRUNC('month', ts) return?
<details><summary>Show answer</summary>A timestamp aligned to the beginning of the month.</details>

### What does timestamp subtraction return?
<details><summary>Show answer</summary>An INTERVAL.</details>

### Why validate text before casting it to DATE?
<details><summary>Show answer</summary>Invalid or unexpected text can fail the cast or be interpreted incorrectly.</details>

### What is TO_CHAR mainly for?
<details><summary>Show answer</summary>Formatting typed values as display text.</details>

### What does TIMESTAMPTZ represent?
<details><summary>Show answer</summary>An instant in time with time-zone-aware semantics.</details>

### Does AT TIME ZONE create a different event?
<details><summary>Show answer</summary>No. It changes the clock representation of the same instant.</details>


# Module 8 Summary

Use this workflow for text and time data:

1. Inspect raw values.
2. Normalize only when the rule is justified.
3. Keep raw values when traceability matters.
4. Use simple text functions before reaching for regex.
5. Separate cleaning from validation.
6. Choose the correct temporal type.
7. Use `EXTRACT` for components.
8. Use `DATE_TRUNC` for time buckets.
9. Subtract timestamps for elapsed intervals.
10. Convert intervals to numeric units only when analysis requires it.
11. Validate text before parsing it into dates.
12. Keep typed dates/timestamps for analysis.
13. Use `TO_CHAR` near the presentation layer.
14. Make time-zone assumptions explicit.
15. Validate transformed values and grouped totals.

## Next
**Module 9: Data Cleaning in PostgreSQL**
